In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load datasets
train = pd.read_csv("../data/train.csv")
calendar = pd.read_csv("../data/calendar_events.csv")
submission_template = pd.read_csv("../data/forecast_submission.csv")

LOOKBACK_DAYS = 28

In [9]:
baseline_per_store = (
    train
    .sort_values("date")
    .groupby("store_id")
    .tail(LOOKBACK_DAYS)
    .groupby("store_id")["revenue"]
    .mean()
)

baseline_per_store

store_id
0     298830.560714
1      35568.902143
2      32421.249643
3      49842.347500
4      19691.897143
5      23838.183571
6      29341.228571
7      29763.415000
8      25148.323214
9      30789.325000
10     22425.690714
Name: revenue, dtype: float64

In [10]:
submission_template = pd.read_csv("../data/forecast_submission.csv")
baseline_submission = submission_template.copy()
baseline_submission["store_id"] = baseline_submission["id"].str.split("_").str[0].astype(int)
baseline_submission["prediction"] = baseline_submission["store_id"].map(baseline_per_store)
baseline_submission = baseline_submission[["id", "prediction"]]
baseline_submission.to_csv("../submissions/baseline_submission.csv", index=False)

In [13]:
train["date"] = pd.to_datetime(train["date"])
calendar["date"] = pd.to_datetime(calendar["date"])
train["weekday"] = train["date"].dt.weekday
calendar["weekday"] = calendar["date"].dt.weekday

In [14]:
train[["date", "weekday"]].head()


,date,weekday
0,2011-01-29,5
1,2011-01-30,6
2,2011-01-31,0
3,2011-02-01,1
4,2011-02-02,2


In [15]:
LOOKBACK_DAYS = 28  # keep same as baseline for a fair comparison

recent = (
    train.sort_values("date")
    .groupby("store_id")
    .tail(LOOKBACK_DAYS)
)

store_weekday_mean = recent.groupby(["store_id", "weekday"])["revenue"].mean()
store_mean = recent.groupby("store_id")["revenue"].mean()  # fallback

store_weekday_mean.head(), store_mean.head()


(store_id  weekday
 0         0          291526.8000
           1          258842.7875
           2          254890.9200
           3          264703.6125
           4          293333.5425
 Name: revenue, dtype: float64,
 store_id
 0    298830.560714
 1     35568.902143
 2     32421.249643
 3     49842.347500
 4     19691.897143
 Name: revenue, dtype: float64)

In [16]:
# Parse store_id and date from submission ids
sub = submission_template.copy()
sub["store_id"] = sub["id"].str.split("_").str[0].astype(int)
sub["date_str"] = sub["id"].str.split("_").str[1]
sub["date"] = pd.to_datetime(sub["date_str"], format="%Y%m%d")

# Add weekday for each future date
sub["weekday"] = sub["date"].dt.weekday

sub[["id", "store_id", "date", "weekday"]].head()


,id,store_id,date,weekday
0,0_20151001,0,2015-10-01,3
1,0_20151002,0,2015-10-02,4
2,0_20151003,0,2015-10-03,5
3,0_20151004,0,2015-10-04,6
4,0_20151005,0,2015-10-05,0


In [17]:
# First try: store+weekday mean
sub["prediction"] = sub.set_index(["store_id", "weekday"]).index.map(store_weekday_mean)

# Fallback: if missing, use store mean
missing = sub["prediction"].isna()
sub.loc[missing, "prediction"] = sub.loc[missing, "store_id"].map(store_mean)

# Final submission format
weekday_submission = sub[["id", "prediction"]].copy()

print("Rows:", len(weekday_submission))
print("Any NaNs?", weekday_submission["prediction"].isna().any())
weekday_submission.head()


Rows: 1012
Any NaNs? False


,id,prediction
0,0_20151001,264703.6125
1,0_20151002,293333.5425
2,0_20151003,361671.2750
3,0_20151004,366844.9875
4,0_20151005,291526.8000


In [18]:
weekday_submission.to_csv("../submissions/weekday_mean_submission.csv", index=False)
print("Saved: submissions/weekday_mean_submission.csv")

Saved: submissions/weekday_mean_submission.csv
